In [1]:
import pandas as pd
import geopandas as gpd 
import requests 
import json
from datetime import datetime
from io import StringIO
import csv
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from shapely import wkt
from scipy.spatial.distance import cdist

import pandas as pd
import geopandas as gpd 
import requests 
import json
from datetime import datetime
from io import StringIO
import csv
import time
import os
import airqualityandclimateAPI
import statsmodels.api as sm

from statsmodels.stats.outliers_influence \
     import variance_inflation_factor as VIF
from statsmodels.stats.anova import anova_lm

from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
                         summarize,
                         poly)
from linearmodels import PanelOLS
import numpy as np

In [2]:
import sklearn.model_selection as skm
import sklearn.linear_model as skl
from sklearn.preprocessing import StandardScaler
from ISLP import load_data
from ISLP.models import (ModelSpec as MS, summarize)
from functools import partial
from sklearn.pipeline import Pipeline
# from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from ISLP.models import \
     (Stepwise,
      sklearn_selected,
      sklearn_selection_path)

In [3]:
import Analysis
from Analysis import model_4_lasso_panel
import Random_Forest

/Users/griffinberonio/Documents/AAE 724/AAE-724-Repo/Analysis.py:48: DtypeWarning: Columns (9,11,12,13,14,15,18,23,28,36,38,40,55,56,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,106) have mixed types. Specify dtype option on import or set low_memory=False.
  climatedata = pd.read_csv(climatedatapath)
/Users/griffinberonio/Documents/AAE 724/AAE-724-Repo/Analysis.py:52: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  airqualitydata = pd.read_csv(airqualitydatapath)
/Users/griffinberonio/Documents/AAE 724/AAE-724-Repo/Analysis.py:71: DtypeWarning: Columns (8,9,10,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  totaldf = pd.read_csv(totaldatapath)


In [4]:
totaldataframe = Analysis.totaldata()

/Users/griffinberonio/Documents/AAE 724/AAE-724-Repo/Analysis.py:89: DtypeWarning: Columns (8,9,10,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  totaldf = pd.read_csv(totaldatapath)


Random Forest PM

In [5]:
random_forest_pm = Random_Forest.random_forest(totaldataframe, pm=True)

Number of NaNs in X: 0
Preprocessing setup complete.
Fitting the Grid Search for RF hyperparameter tuning...
100
Fitting the RF model
(51082, 167)
                                             feature  importance
112                       SO2 Mass (short tons)_lag2    0.014460
139                  DailyPeakWindDirection_cos_lag3    0.013461
4         DailyDepartureFromNormalAverageTemperature    0.013183
80                                         DATE_lag2    0.012534
130  DailyDepartureFromNormalAverageTemperature_lag3    0.011551
114                       NOx Mass (short tons)_lag2    0.011299
156                       CO2 Mass (short tons)_lag3    0.010971
98              DailySustainedWindDirection_cos_lag2    0.010831
36                                       sunrise_sin    0.010700
10                                           Sunrise    0.010583
29                                  Gross Load (MWh)    0.010549
44   DailyDepartureFromNormalAverageTemperature_lag1    0.010475
0       

Random Forest AQI:

In [6]:
random_forest_aqi = Random_Forest.random_forest(totaldataframe, pm=False)


Number of NaNs in X: 0
Preprocessing setup complete.
Fitting the Grid Search for RF hyperparameter tuning...
100
Fitting the RF model
(51082, 167)
                                             feature  importance
4         DailyDepartureFromNormalAverageTemperature    0.016058
44   DailyDepartureFromNormalAverageTemperature_lag1    0.015018
87   DailyDepartureFromNormalAverageTemperature_lag2    0.013881
53                   DailyPeakWindDirection_cos_lag1    0.013785
130  DailyDepartureFromNormalAverageTemperature_lag3    0.013110
98              DailySustainedWindDirection_cos_lag2    0.012847
139                  DailyPeakWindDirection_cos_lag3    0.012210
96                   DailyPeakWindDirection_cos_lag2    0.011879
55              DailySustainedWindDirection_cos_lag1    0.011819
15                   DailySustainedWindDirection_cos    0.011058
141             DailySustainedWindDirection_cos_lag3    0.010765
88               DailyMaximumDryBulbTemperature_lag2    0.010588
80      

X matrices:

In [14]:
features_pm = random_forest_pm[3].head(20)
features_aqi = random_forest_aqi[3].head(20)


In [15]:
x_rf_pm = list(features_pm['feature'])
x_rf_aqi = list(features_aqi['feature'])

In [16]:
# Y vars 

fe = ['CLIMATE_STATION_NAME','YEAR']

y_pm = ['arithmetic_mean']
y_aqi = ['aqi']

# xvars = ['Number', 'Capacity']
xvarsnum = ['Number']
xvarscap = ['Capacity']

## X variables using the PM results: 
x_pm_num = xvarsnum + x_rf_pm
x_pm_cap = xvarscap + x_rf_pm

## X variables using the AQIQ results:
x_aqi_num = xvarsnum + x_rf_aqi
x_aqi_cap = xvarscap + x_rf_aqi

In [17]:
PM_RF_NUM = model_4_lasso_panel(totaldataframe, x_pm_num, y_pm, fe)
PM_RF_NUM

/Users/griffinberonio/Documents/AAE 724/AAE-724-Repo/Analysis.py:817: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

Number, SO2 Mass (short tons)_lag2, DailyPeakWindDirection_cos_lag3, DailyDepartureFromNormalAverageTemperature, DailyDepartureFromNormalAverageTemperature_lag3, NOx Mass (short tons)_lag2, CO2 Mass (short tons)_lag3, DailySustainedWindDirection_cos_lag2, sunrise_sin, Sunrise, Gross Load (MWh), DailyDepartureFromNormalAverageTemperature_lag1, CO2 Mass (short tons), sunrise_sin_lag1, DailyPeakWindDirection_cos, NOx Mass (short tons)_lag1

  results = model.fit(cov_type='clustered', cluster_entity=True)


Dep. Variable:,arithmetic_mean,R-squared:,0.0002
Estimator:,PanelOLS,R-squared (Between):,-0.2398
No. Observations:,72975,R-squared (Within):,-0.0017
Date:,"Fri, May 01 2026",R-squared (Overall):,-0.0019
Time:,20:48:33,Log-likelihood,-2.371e+05
Cov. Estimator:,Clustered,,
,,F-statistic:,3.4365
Entities:,2,P-value,0.0082
Avg Obs:,3.649e+04,Distribution:,"F(4,72960)"
Min Obs:,3.574e+04,,
Max Obs:,3.723e+04,F-statistic (robust):,1823.8


In [19]:
PM_RF_CAP = model_4_lasso_panel(totaldataframe, x_pm_cap, y_pm, fe)
PM_RF_CAP

/Users/griffinberonio/Documents/AAE 724/AAE-724-Repo/Analysis.py:817: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

Capacity, SO2 Mass (short tons)_lag2, DailyPeakWindDirection_cos_lag3, DailyDepartureFromNormalAverageTemperature, DailyDepartureFromNormalAverageTemperature_lag3, NOx Mass (short tons)_lag2, CO2 Mass (short tons)_lag3, DailySustainedWindDirection_cos_lag2, sunrise_sin, Sunrise, Gross Load (MWh), DailyDepartureFromNormalAverageTemperature_lag1, CO2 Mass (short tons), sunrise_sin_lag1, DailyPeakWindDirection_cos, NOx Mass (short tons)_lag1

  results = model.fit(cov_type='clustered', cluster_entity=True)


Dep. Variable:,arithmetic_mean,R-squared:,0.0002
Estimator:,PanelOLS,R-squared (Between):,-0.2398
No. Observations:,72975,R-squared (Within):,-0.0017
Date:,"Fri, May 01 2026",R-squared (Overall):,-0.0019
Time:,20:50:37,Log-likelihood,-2.371e+05
Cov. Estimator:,Clustered,,
,,F-statistic:,3.4365
Entities:,2,P-value,0.0082
Avg Obs:,3.649e+04,Distribution:,"F(4,72960)"
Min Obs:,3.574e+04,,
Max Obs:,3.723e+04,F-statistic (robust):,1823.8


In [20]:
AQI_RF_NUM = model_4_lasso_panel(totaldataframe, x_aqi_num, y_aqi, fe)
AQI_RF_NUM

/Users/griffinberonio/Documents/AAE 724/AAE-724-Repo/Analysis.py:817: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

Number, DailyDepartureFromNormalAverageTemperature, DailyDepartureFromNormalAverageTemperature_lag1, DailyDepartureFromNormalAverageTemperature_lag2, DailyPeakWindDirection_cos_lag1, DailyDepartureFromNormalAverageTemperature_lag3, DailySustainedWindDirection_cos_lag2, DailyPeakWindDirection_cos_lag3, DailyPeakWindDirection_cos_lag2, DailySustainedWindDirection_cos_lag1, DailySustainedWindDirection_cos_lag3, DailyMaximumDryBulbTemperature_lag2, DailyMaximumDryBulbTemperature_lag3, DailyMaximumDryBulbTemperature, DailyMaximumDryBulbTemperature_lag1, Gross Load (MWh)_lag3, Gross Load (MWh)_lag2, SO2 Mass (short tons)_lag1

  results = model.fit(cov_type='clustered', cluster_entity=True)


Dep. Variable:,aqi,R-squared:,9.797e-05
Estimator:,PanelOLS,R-squared (Between):,-0.2183
No. Observations:,72975,R-squared (Within):,-0.0152
Date:,"Fri, May 01 2026",R-squared (Overall):,-0.0158
Time:,20:51:11,Log-likelihood,-3.103e+05
Cov. Estimator:,Clustered,,
,,F-statistic:,3.5743
Entities:,2,P-value,0.0280
Avg Obs:,3.649e+04,Distribution:,"F(2,72962)"
Min Obs:,3.574e+04,,
Max Obs:,3.723e+04,F-statistic (robust):,1.148e+04
